# Графовые нейронные сети: обучение на данных со структурой связей

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Графовые нейронные сети».

## Что делает метод

Многие научные объекты естественно описываются графом — набором узлов и связей: атомы и химические связи в молекуле, белки и их взаимодействия, кристаллическая структура материала. Графовая нейронная сеть учится работать прямо с такой структурой, не разрушая её.

Сеть работает по принципу передачи сообщений: на каждом слое узел обновляет своё представление, агрегируя информацию от соседей. После нескольких слоёв представление каждого узла отражает его локальное окружение. В этом блокноте мы реализуем компактную графовую свёрточную сеть (GCN) и применим её к классификации узлов. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы хотите охарактеризовать человека в социальной сети. Можно смотреть только на его личные данные — но гораздо точнее учитывать его круг общения: с кем он дружит, кто его коллеги. Сеть коллег тоже не случайна — и так далее по цепочке.

Графовая нейронная сеть работает по тому же принципу. Каждый узел (атом, белок, регион, человек) имеет свои признаки — но ГНС умеет учитывать и соседей. На каждом слое сети каждый узел «спрашивает» своих соседей: «что вы знаете?», собирает их ответы и обновляет своё представление. Через два слоя узел знает про соседей своих соседей. Это называется **передача сообщений** (message passing).

Для учёного это означает:
- Молекула описывается не вектором абстрактных дескрипторов, а графом атомов и связей — сеть сама извлекает признаки.
- Белок в сети взаимодействий получает представление, учитывающее его партнёров — а не только собственную последовательность.
- Кристалл описывается как граф атомов с учётом координационного окружения.

Ключевое отличие от обычных нейронных сетей: одна и та же операция применяется к каждому узлу независимо от его положения в графе и числа соседей. Это делает ГНС инвариантными к перестановке узлов.

Терминологический минимум для этого блокнота:
- **Узел** (node) — объект в графе (атом, белок, наблюдение).
- **Ребро** (edge) — связь между двумя узлами (химическая связь, взаимодействие).
- **Матрица смежности** (adjacency matrix) — квадратная матрица, где 1 означает наличие ребра между узлами.
- **GCN** (Graph Convolutional Network) — графовая свёрточная сеть: простейший вариант ГНС, усредняющий признаки соседей.
- **Нормировка по степени** — нормировка агрегированных признаков на число связей узла. Без неё узлы с разным числом связей несравнимы.
- **Функция потерь / потеря** — число, которое сеть минимизирует при обучении (здесь: кросс-энтропия классификации).

## 1. Установка библиотек

Чтобы блокнот запускался без сложной настройки окружения, графовый слой реализован средствами базового `torch` — специализированные графовые пакеты не требуются.

In [ ]:
%pip install -q torch numpy scikit-learn matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сформируем синтетический граф из трёх сообществ: узлы внутри одного сообщества связаны плотно, между сообществами — редко. У каждого узла есть слабо информативные собственные признаки. Задача — определить сообщество узла; её можно надёжно решить, только используя структуру связей. Это модель типичной задачи классификации узлов (например, по сети взаимодействий).

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n_communities = 3
n_per = 40
n_nodes = n_communities * n_per
labels = np.repeat(np.arange(n_communities), n_per)

# Матрица смежности: плотные связи внутри сообществ, редкие - между ними.
p_in, p_out = 0.35, 0.03
A = np.zeros((n_nodes, n_nodes), dtype="float32")
for i in range(n_nodes):
    for j in range(i + 1, n_nodes):
        p = p_in if labels[i] == labels[j] else p_out
        if rng.random() < p:
            A[i, j] = A[j, i] = 1.0

# Признаки узлов: слабый сигнал плюс заметный шум.
centers = rng.normal(0, 0.4, (n_communities, 8))
features = (centers[labels] + rng.normal(0, 1.0, (n_nodes, 8))).astype("float32")

print(f"Узлов: {n_nodes}, рёбер: {int(A.sum() / 2)}, сообществ: {n_communities}")

## 4. Применение метода

### Шаг 1. Нормировка матрицы смежности и архитектура GCN

**Зачем нормировать матрицу смежности?** Если просто суммировать признаки соседей, узлы с большим числом связей получат доминирующие представления. Стандартная нормировка GCN добавляет петли (узел учитывает себя) и делит на геометрическое среднее степеней смежных узлов: `A_hat = D^(-1/2) (A + I) D^(-1/2)`. После этого каждый узел получает взвешенное среднее своих соседей и себя.

**Как работает слой GCN?** Формула проста: `H = A_hat @ X @ W + b`. Матричное умножение на `A_hat` слева — это агрегация признаков соседей. Умножение на `W` справа — обычный линейный слой, обучаемый как в стандартной нейросети.

Двухслойная сеть: первый слой агрегирует соседей (радиус 1), второй — соседей соседей (радиус 2).

**Полуконтролируемое обучение:** мы обучаем сеть только на половине узлов, а тестируем на другой. Но граф виден целиком при вычислении — это и есть ключевое преимущество: информация от всех узлов через рёбра проникает даже к узлам, чьи метки скрыты.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# Нормировка смежности: A_hat = D^(-1/2) (A + I) D^(-1/2).
A_loops = A + np.eye(n_nodes, dtype="float32")
deg = A_loops.sum(axis=1)
d_inv_sqrt = np.diag(1.0 / np.sqrt(deg))
A_norm = torch.from_numpy(d_inv_sqrt @ A_loops @ d_inv_sqrt)
X = torch.from_numpy(features)
y = torch.from_numpy(labels).long()


class GCNLayer(nn.Module):
    """Графовый свёрточный слой: агрегация признаков соседей."""

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, a_norm):
        return self.linear(a_norm @ x)           # сообщение от соседей


class GCN(nn.Module):
    """Двухслойная графовая свёрточная сеть для классификации узлов."""

    def __init__(self, in_dim, hidden, n_classes):
        super().__init__()
        self.gc1 = GCNLayer(in_dim, hidden)
        self.gc2 = GCNLayer(hidden, n_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, a_norm):
        h = F.relu(self.gc1(x, a_norm))
        h = self.dropout(h)
        return self.gc2(h, a_norm)


model = GCN(in_dim=8, hidden=16, n_classes=n_communities)
print(model)

### Шаг 2. Обучение и сравнение с базовой моделью

Обучим GCN 120 эпох. Функция потерь — кросс-энтропия: сеть учится правильно классифицировать узлы на обучающем наборе.

Для честного сравнения обучим базовую модель — логистическую регрессию только на признаках узлов (без информации о связях). Если GCN превосходит её заметно — значит, структура графа несёт дополнительную информацию о принадлежности узла к сообществу.

Это именно тот случай, когда собственных признаков узлов недостаточно (мы добавили много шума), а структура связей (плотные сообщества) надёжно указывает на принадлежность.

In [ ]:
# Разбиение узлов на обучающие и тестовые (часть узлов скрыта при обучении).
perm = rng.permutation(n_nodes)
train_idx = torch.from_numpy(perm[:n_nodes // 2]).long()
test_idx = torch.from_numpy(perm[n_nodes // 2:]).long()

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

history = {"loss": [], "test_acc": []}
n_epochs = 120
for epoch in range(1, n_epochs + 1):
    model.train()
    optimizer.zero_grad()
    out = model(X, A_norm)
    loss = criterion(out[train_idx], y[train_idx])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        pred = model(X, A_norm).argmax(dim=1)
        test_acc = (pred[test_idx] == y[test_idx]).float().mean().item()
    history["loss"].append(loss.item())
    history["test_acc"].append(test_acc)
    if epoch % 20 == 0 or epoch == 1:
        print(f"Эпоха {epoch:3d}: потеря {loss.item():.4f}, "
              f"доля верных на тесте {test_acc:.3f}")

In [ ]:
# Сравнение с моделью, игнорирующей структуру связей (только признаки узлов).
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(features[train_idx.numpy()], labels[train_idx.numpy()])
baseline_acc = clf.score(features[test_idx.numpy()], labels[test_idx.numpy()])
print(f"Доля верных, графовая сеть:        {history['test_acc'][-1]:.3f}")
print(f"Доля верных, только по признакам:  {baseline_acc:.3f}")

### Шаг 3. Визуализация результата

Два графика показывают, как обучалась сеть и что она «узнала» о структуре данных.

**Левый:** ход обучения — функция потерь и доля верных ответов на тестовых узлах по эпохам.

**Правый:** представления узлов из первого слоя GCN, спроецированные в 2D через PCA. Это «внутренняя карта» графа, построенная сетью. Цвет — истинное сообщество узла.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.3))

# Ход обучения
epochs = range(1, n_epochs + 1)
axes[0].plot(epochs, history["loss"], color=VIZ["series"][0],
             label="Функция потерь")
axes[0].set_xlabel("Эпоха")
axes[0].set_ylabel("Функция потерь")
ax_acc = axes[0].twinx()
ax_acc.plot(epochs, history["test_acc"], color=VIZ["series"][1],
            label="Доля верных (тест)")
ax_acc.set_ylabel("Доля верных ответов")
ax_acc.grid(False)
axes[0].set_title("Ход обучения")
lines = axes[0].get_lines() + ax_acc.get_lines()
axes[0].legend(lines, [ln.get_label() for ln in lines], loc="center right")

# Представления узлов
model.eval()
with torch.no_grad():
    embeddings = F.relu(model.gc1(X, A_norm)).numpy()
coords = PCA(n_components=2, random_state=42).fit_transform(embeddings)
palette = get_palette(n_communities)
for c in range(n_communities):
    mask = labels == c
    axes[1].scatter(coords[mask, 0], coords[mask, 1], s=42, alpha=0.85,
                    color=palette[c], label=f"Сообщество {c + 1}")
axes[1].set_title("Представления узлов из графовой сети")
axes[1].set_xlabel("Ось 1")
axes[1].set_ylabel("Ось 2")
axes[1].legend(loc="best")

fig.tight_layout()
plt.show()

**Как читать эти графики:**

- **Левый — ход обучения:** синяя кривая (потеря) должна убывать, зелёная (точность) — расти и выйти на плато. Если точность остаётся низкой — граф слишком слабо структурирован или сеть слишком мала.
- **Правый — карта представлений:** если GCN успешно использовала структуру графа, три цвета образуют три отдельных компактных кластера. Чем лучше они разделены — тем лучше сеть «разобралась» в сообществах. Сравните с исходными признаками: если там три цвета перемешаны, а здесь разделены — это прямое подтверждение, что структура графа помогла.

Обратите внимание на числа под графиком из ячейки выше: насколько доля верных ответов GCN превышает базовую модель (только признаки)?

## 5. Интерпретация результата

- **Доля верных ответов**: сравнение с моделью, использующей только признаки узлов, показывает вклад структуры связей. Заметный выигрыш графовой сети означает, что информация о соседстве действительно полезна для задачи.
- **Ход обучения**: функция потерь снижается, доля верных растёт и выходит на плато.
- **Карта представлений**: если сеть успешно использовала структуру, узлы одного сообщества образуют компактные, разделённые группы — даже когда их собственные признаки слабо различимы.

Помните о практических ограничениях: при большом числе слоёв представления узлов становятся неразличимыми (oversmoothing) — обычно достаточно двух-четырёх слоёв. Разбиение на обучение и тест для графов должно учитывать структурное сходство, иначе точность будет завышена.

## 6. Попробуйте на своих данных

Для своего графа подготовьте: матрицу смежности `A` размера `(узлы, узлы)` со значениями 0 и 1, матрицу признаков узлов `features` и вектор меток `labels`.

1. Загрузите файлы графа в Colab через панель «Файлы» (например, список рёбер и таблицу признаков).
2. Снимите комментарии в ячейке ниже и соберите матрицу смежности из списка рёбер.
3. Для крупных графов и более сложных задач (предсказание свойств рёбер или графа целиком) удобны специализированные библиотеки графового обучения.
4. Последовательно выполните ячейки разделов 4 и 5.

## 7. Поэкспериментируйте сами

Измените один параметр и перезапустите ячейки раздела 3–5:

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `p_in = 0.2` | Снизить плотность связей внутри сообщества | Граф менее структурирован — падает ли преимущество GCN над базовой? |
| `p_out = 0.1` | Увеличить связи между сообществами | Граф «перепутан» — труднее ли теперь задача? |
| `hidden = 32` | Увеличить скрытый слой | Улучшается ли точность? |
| `n_communities = 4` | Добавить четвёртое сообщество | Нужно также изменить `n_classes=4` в `GCN(...)` |
| Добавить третий слой GCN | Добавить `self.gc3 = GCNLayer(n_classes, n_classes)` | Помогают ли три слоя? Иногда наоборот хуже — это «oversmoothing» |

Дополнительно: можно визуализировать исходные признаки узлов (до GCN) с теми же цветами и сравнить с картой после GCN — разница наглядно покажет вклад структуры.

In [ ]:
# --- Шаблон загрузки своего графа ---
# import pandas as pd
#
# edges = pd.read_csv("edges.csv")              # столбцы: source, target
# node_feats = pd.read_csv("node_features.csv") # признаки узлов, по строке на узел
# node_labels = pd.read_csv("node_labels.csv")  # столбец с меткой класса
#
# n_nodes = len(node_feats)
# A = np.zeros((n_nodes, n_nodes), dtype="float32")
# for s, t in edges[["source", "target"]].itertuples(index=False):
#     A[s, t] = A[t, s] = 1.0
# features = node_feats.select_dtypes(include="number").to_numpy("float32")
# labels = node_labels.iloc[:, 0].astype("category").cat.codes.to_numpy()
#
# print(f"Узлов: {n_nodes}, рёбер: {int(A.sum() / 2)}")
# Далее повторите ячейки раздела 4.

## 8. Краткие выводы

- Графовые нейронные сети работают на данных со структурой связей: молекулах, сетях взаимодействий, кристаллах.
- Ключевая операция — передача сообщений: каждый узел обновляет своё представление, агрегируя информацию от соседей.
- Нормировка матрицы смежности необходима: без неё узлы с разным числом связей несравнимы.
- Два слоя позволяют учитывать окружение радиуса 2. Большое число слоёв ведёт к «oversmoothing» — все представления сливаются.
- Сравнение с базовой моделью (только признаки) — обязательный тест: если GCN не превосходит, связи не несут информации о задаче.
- Для крупных графов и сложных задач используют специализированные библиотеки: PyTorch Geometric, DGL. Для молекул — эквивариантные архитектуры (e3nn, DimeNet).

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На карте представлений узлов (PCA после первого слоя GCN) три сообщества образовали хорошо разделённые кластеры, хотя собственные признаки узлов были сильно зашумлены. Что именно позволило сети разделить их, если не сами признаки?
2. Вы добавляете третий слой GCN поверх двух существующих и замечаете, что точность на тесте упала. Как называется это явление и какова его причина?
3. Вы применяете GCN к реальной сети цитирований статей: узлы — статьи, рёбра — ссылки. При случайном разбиении узлов на обучение и тест модель показывает 0.92, а при разбиении, исключающем связи между обучающей и тестовой частями (scaffold split), — 0.74. Что объясняет эту разницу?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Структура рёбер: внутри каждого сообщества связи плотные, между сообществами — редкие. При агрегации на каждом слое узел усредняет представления своих соседей; поскольку соседи принадлежат тому же сообществу, их признаки сходятся, а представления разных сообществ расходятся — даже если начальные признаки узлов почти неразличимы.
2. Это over-smoothing (избыточное сглаживание): с каждым слоем GCN каждый узел смешивает информацию из всё более широкого окружения, и в итоге представления всех узлов сходятся к общему среднему по графу, теряя индивидуальность.
3. При случайном разбиении тестовые узлы имеют рёбра к обучающим: сеть «видит» часть тестовых данных через структуру графа во время обучения — это утечка (graph data leakage). Scaffold split исключает такие прямые связи и даёт честную оценку обобщения на узлы из структурно независимого подграфа.
</details>
